# atomic-povray Prototype 1

The geometry stage is deliberately separate. Run it once, then reuse `geometry` while experimenting with style and camera settings. The Fe–O rule below uses the default asymmetric boundary behavior: primary Fe atoms may add bonded O extension atoms, and those extension atoms never seed further searches.

In [ ]:
from pathlib import Path
from atomic_povray import *

project = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
poscar = project / 'tests' / 'data' / 'hematite_1x1_unrelaxed_bare.vasp'
povray = r'C:\Program Files\POV-Ray\v3.8-beta\bin\pvengine64.exe'

In [ ]:
structure = load_structure(poscar)
geometry = build_geometry(
    structure,
    repetitions=(2, 1, 1),
    bounds=CartesianBounds(z_min=17.0),
    bond_rules=(BondRule('Fe', 'O', 0.1, 2.45),),
)
len(geometry.primary_atoms), len(geometry.extension_atoms), len(geometry.bonds)

## Legacy colors, sizes, and finishes

These values are direct translations of `global_colors_sizes.pov`. POV-Ray permits RGB components above 1, so the oxygen and iron colors are intentionally `(1.05, 0.10, 0.05)` and `(0.10, 0.10, 1.10)`. The historical `sizeFact` is 0.85; keeping it as one variable makes the multiplication visible.

In [ ]:
size_factor = 0.85
legacy_colors = {
    'H': Color(0.90, 0.90, 0.90),
    'C': Color(0.20, 0.20, 0.20),
    'O': Color(1.05, 0.10, 0.05),
    'Fe': Color(0.10, 0.10, 1.10),
    'Rh': Color(0.50, 0.50, 0.50),
    'Pt': Color(0.60, 0.60, 0.60),
    'O_ad': Color(1.00, 0.33, 0.00),
}
legacy_radii = {
    'H': 0.20 * size_factor,
    'C': 0.40 * size_factor,
    'O': 0.40 * size_factor,
    'Fe': 0.65 * size_factor,
    'Rh': 0.65 * size_factor,
    'Pt': 0.75 * size_factor,
}

def sphere_style(symbol):
    color = legacy_colors[symbol]
    return AtomStyle(
        legacy_radii[symbol],
        color,
        Material(color, ambient=0.10, diffuse=0.60, phong=0.30, phong_size=10),
    )

bond_finish = Material(
    Color(0, 0, 0), ambient=0.10, diffuse=0.60, phong=0.0, phong_size=10
)
styles = StyleConfig(
    elements={symbol: sphere_style(symbol) for symbol in ('Fe', 'O')},
    bonds={
        'Fe-O': BondStyle(radius=0.074, material_template=bond_finish),
    },
)
styled = apply_styles(geometry, styles)

## Legacy camera and lighting

The light settings below translate `global_lighting_side.pov`: position `(-4000, -6000, 6000)`, intensity 0.9, a 9×9 soft source spanning 35°, adaptive level 3, global ambient light 0.1, and sphere Phong parameters 0.3/10. The camera remains the side-view camera already reconstructed from the original scene.

In [ ]:
camera = Camera.orthographic(
    location=(5.0, -100.0, 25.5),
    target=(5.0, 0.0, 25.5),
    up=(0.0, 0.0, 1.0),
    width=21.0,
)
legacy_light = AreaLight(
    location=(-4000.0, -6000.0, 6000.0),
    target=camera.target,
    intensity=0.9,
    angular_diameter=35.0,
    samples=(9, 9),
    adaptive=3,
)
scene = make_scene(
    styled.primitives,
    camera=camera,
    lights=(legacy_light,),
    ambient_light=Color(0.10, 0.10, 0.10),
    background=Background(Color(1.0, 1.0, 1.0, alpha=0.0)),
)

## Render with the legacy INI settings

This uses the original 1024×768 output, quality 5 (required for the area light), gamma 2.0, recursive antialiasing with threshold 0.05, PNG output, and an alpha channel. Once this regression render matches, width and height can be changed independently.

In [ ]:
render_config = RenderConfig(
    width=1024,
    height=768,
    quality=5,
    antialias=True,
    antialias_threshold=0.05,
    sampling_method=2,
    display_gamma=2.0,
    file_gamma=2.0,
    transparent=True,
    display=True,
    executable=povray,
    povray_version='3.8',
)
result = render_scene(
    scene,
    project / 'hematite_legacy_settings.png',
    render_config,
)
result.image_path

## Export the POV and INI files without rendering

`write_scene` exports only SDL. `write_ini` writes the matching render settings separately, which is useful when opening the files in the POV-Ray GUI. Render the `.ini` file—not the `.pov` alone—so the gamma, antialiasing, transparency, quality, and resolution settings are retained.

In [ ]:
scene_path = write_scene(
    scene,
    project / 'hematite_notebook.pov',
    width=render_config.width,
    height=render_config.height,
    povray_version=render_config.povray_version,
)
ini_path = write_ini(
    scene_path,
    project / 'hematite_notebook.png',
    render_config,
)
scene_path, ini_path